# Task 2: Exploratory Data Analysis & Integrity Cleaning Pipeline
**Author:** Data Analyst Intern

--- 

## Executive Summary & Workflow
This notebook implements an enterprise-grade, modular Data Cleaning & Exploratory Data Analysis (EDA) Pipeline designed for uncurated industry datasets. 

### Key Highlights:
1. **Automated Anomaly & Structural Diagnosis**: Automated detection of missingness, data types, and structural inconsistencies.
2. **Outlier Mitigation via IQR Capping**: Utilizes Interquartile Range (IQR) capping (Winsorization) to restrict extreme variance without destroying data records.
3. **Systematic Imputation**: Robust imputation using **Median** (for continuous variables sensitive to skewness) and **Mode** (for categorical variables).
4. **Descriptive Distribution Analysis**: Clear, production-quality visual plots (KDE Histograms + Boxplots) to evaluate distribution post-cleaning.

In [ ]:
# Imports and global plotting configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Professional Styling Setup
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 120

## 1. Modular Data Pipeline Class
Below is the complete `DataIntegrityPipeline` class encapsulating data loading, anomaly diagnostics, outlier treatment, systematic imputation, and distribution visualization.

In [ ]:
class DataIntegrityPipeline:
    """
    Production-grade Data Cleaning & EDA Pipeline.
    """
    def __init__(self, df: pd.DataFrame):
        self.raw_df = df.copy()
        self.df = df.copy()
        
    def diagnose_data(self):
        """Diagnoses structural properties, data types, and missing value ratios."""
        print("="*55)
        print("STEP 1: DATASET DIAGNOSTICS & ABNORMALITY DETECTION")
        print("="*55)
        print(f"Shape: {self.df.shape[0]} Rows, {self.df.shape[1]} Columns\n")
        
        missing_summary = pd.DataFrame({
            'Missing Values': self.df.isnull().sum(),
            'Percentage (%)': (self.df.isnull().sum() / len(self.df) * 100).round(2),
            'Data Type': self.df.dtypes
        })
        
        print("--- Missing Values & Data Types ---")
        print(missing_summary)
        print("\n")
        return missing_summary

    def handle_outliers_iqr(self, columns: list = None, factor: float = 1.5):
        """
        Detects and caps outliers using the Interquartile Range (IQR) method.
        Outliers are capped at lower and upper boundaries to retain row sample size.
        """
        print("="*55)
        print("STEP 2: OUTLIER TREATMENT (IQR CAPPING / WINSORIZATION)")
        print("="*55)
        
        if columns is None:
            columns = self.df.select_dtypes(include=[np.number]).columns.tolist()
            
        for col in columns:
            if col in self.df.columns and np.issubdtype(self.df[col].dtype, np.number):
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - (factor * IQR)
                upper_bound = Q3 + (factor * IQR)
                
                outliers_mask = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
                outliers_count = outliers_mask.sum()
                
                # Capping values
                self.df[col] = np.clip(self.df[col], lower_bound, upper_bound)
                print(f"Column '{col}': {outliers_count} outliers capped to range [{lower_bound:.2f}, {upper_bound:.2f}]")
        print("\n")

    def handle_missing_values(self, num_strategy: str = 'median', cat_strategy: str = 'mode'):
        """
        Systematically imputes missing values:
        - Numerical: Median (robust to skewness) or Mean
        - Categorical: Mode (most frequent value)
        """
        print("="*55)
        print("STEP 3: SYSTEMATIC MISSING VALUE IMPUTATION")
        print("="*55)
        
        # Numerical Imputation
        num_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            missing_cnt = self.df[col].isnull().sum()
            if missing_cnt > 0:
                val = self.df[col].median() if num_strategy == 'median' else self.df[col].mean()
                self.df[col].fillna(val, inplace=True)
                print(f"Imputed {missing_cnt} missing values in Continuous '{col}' using {num_strategy.capitalize()} ({val:.2f})")
                
        # Categorical Imputation
        cat_cols = self.df.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            missing_cnt = self.df[col].isnull().sum()
            if missing_cnt > 0:
                val = self.df[col].mode()[0]
                self.df[col].fillna(val, inplace=True)
                print(f"Imputed {missing_cnt} missing values in Categorical '{col}' using Mode ('{val}')")
        print("\n")

    def plot_distributions(self, columns: list = None):
        """
        Generates Histogram with KDE overlay and Boxplots for numeric variable evaluation.
        """
        print("="*55)
        print("STEP 4: EXPLORATORY DATA ANALYSIS (DESCRIPTIVE DISTRIBUTIONS)")
        print("="*55)
        
        if columns is None:
            columns = self.df.select_dtypes(include=[np.number]).columns.tolist()
            
        num_vars = len(columns)
        if num_vars == 0:
            print("No numerical columns available for distribution plotting.")
            return
            
        fig, axes = plt.subplots(num_vars, 2, figsize=(14, 3.8 * num_vars))
        if num_vars == 1:
            axes = np.expand_dims(axes, axis=0)
            
        fig.suptitle("Descriptive Variable Distributions & Boxplot Analysis (Post-Pipeline)", fontsize=15, fontweight='bold', y=0.99)
        
        for i, col in enumerate(columns):
            # Histogram + KDE
            sns.histplot(self.df[col], kde=True, ax=axes[i, 0], color="#2b5c8f", bins=25, edgecolor='black', alpha=0.6)
            axes[i, 0].set_title(f"Distribution & Density: {col}", fontsize=11, fontweight='bold')
            axes[i, 0].set_xlabel(col)
            axes[i, 0].set_ylabel("Frequency")
            
            # Boxplot
            sns.boxplot(x=self.df[col], ax=axes[i, 1], color="#e07a5f", fliersize=4)
            axes[i, 1].set_title(f"Boxplot (Outlier Check): {col}", fontsize=11, fontweight='bold')
            axes[i, 1].set_xlabel(col)
            
        plt.tight_layout()
        plt.show()
        
    def get_processed_data(self) -> pd.DataFrame:
        return self.df

## 2. Dataset Initialization & Execution
*Note: Replace the synthetic industry dataset below with your target CSV dataset file path (e.g., `pd.read_csv('your_dataset.csv')`).*

In [ ]:
# Generating a synthetic raw industry sample dataset to demonstrate functionality
np.random.seed(42)
n_samples = 250

raw_data = {
    'Employee_ID': range(1001, 1001 + n_samples),
    'Age': np.random.choice([22, 25, 28, 32, 38, 45, 52, np.nan, 115, 125], size=n_samples, p=[0.2, 0.2, 0.2, 0.15, 0.1, 0.08, 0.04, 0.02, 0.005, 0.005]),
    'Annual_Income_k': np.random.choice([45, 55, 65, 80, 95, 110, np.nan, 450, 600], size=n_samples, p=[0.25, 0.25, 0.2, 0.12, 0.08, 0.05, 0.03, 0.01, 0.01]),
    'Performance_Score': np.random.choice([1.2, 2.5, 3.0, 3.8, 4.5, 4.9, np.nan], size=n_samples, p=[0.05, 0.15, 0.35, 0.3, 0.1, 0.02, 0.03]),
    'Department': np.random.choice(['Engineering', 'Data Science', 'Sales', 'HR', np.nan], size=n_samples, p=[0.35, 0.25, 0.2, 0.15, 0.05])
}

raw_df = pd.DataFrame(raw_data)

# Initialize Pipeline
pipeline = DataIntegrityPipeline(raw_df)

# Execute Pipeline Steps
pipeline.diagnose_data()
pipeline.handle_outliers_iqr(columns=['Age', 'Annual_Income_k', 'Performance_Score'])
pipeline.handle_missing_values(num_strategy='median', cat_strategy='mode')
pipeline.plot_distributions(columns=['Age', 'Annual_Income_k', 'Performance_Score'])

# Extract Cleaned Dataset
cleaned_df = pipeline.get_processed_data()

## 3. Post-Cleaning Summary & Inspection
Verifying that all missing values have been eliminated and extreme values capped properly.

In [ ]:
print("Processed Dataset Sample:")
display(cleaned_df.head(10))

print("\nFinal Summary Statistics:")
display(cleaned_df.describe().T)